# Sampling Method Comparison

**What's in this notebook?** This notebook benchmarks and compares the different approaches for finding Type IIB SUSY flux vacua in JAXVacua on the CP⁴[1,1,1,6,9][18] model ($h^{1,2}=2$).

- **Model setup** — standard test model matching arXiv:2501.03984.
- **Bounding box analysis** — how eigenvalue bounds determine the search space and why it grows rapidly with $N_{\max}$.
- **Method overview** — four approaches: manual ISD, `enumerate_fluxes`, `sample_bounded_fluxes` (Gaussian prior), and `sample_bounded_fluxes` with `linearised_shifts`.
- **Head-to-head benchmark** — timing, yield, and rate comparison at $N_{\max}=10$ (small) and $N_{\max}=200$ (large).
- **Gaussian prior analysis** — why $h \sim \mathcal{N}(0, \sigma^2 M)$ outperforms uniform sampling by 50×.
- **Summary and recommendations** — which method to use when.

(*Created:* Andreas Schachner, 2026-03-25)

## Imports

In [ ]:
import warnings
import numpy as np
import time, math

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt

import jaxvacua as jvc
from jaxvacua.flux_bounding import bounded_fluxes

warnings.filterwarnings('ignore')

## 1. Model setup

We use the CP⁴[1,1,1,6,9][18] model at LCS with $h^{1,2}=2$. This is the standard test model from arXiv:2501.03984 (Dataset B).

In [ ]:
h12 = 2
model = jvc.FluxVacuaFinder(h12=h12, model_ID=1, model_type="KS", maximum_degree=5)
model.lcs_tree.a_matrix = jnp.array([[4.5, 1.5], [1.5, 0.]])

n_fl = model.n_fluxes  # = h12+1 = 3 per half, 6 total

print(f"Model: h12={h12}, n_fluxes={n_fl}")
print(f"  ISD matrix: {n_fl}×{n_fl}")
print(f"  Gauge kinetic matrix: {model.dimension_H3}×{model.dimension_H3}")
print(f"  Flux vector: [f|h] of length {2*n_fl}")

## 2. Bounding box scaling with $N_{\max}$

The eigenvalue-based bounding box from arXiv:2501.03984 defines the search region for NSNS-fluxes $h = (h_1, h_2)$. The box dimensions are:

$$h_{1,\text{box}} = \sqrt{\frac{N_{\max}}{s_{\min}\,\tilde\mu_{\min}}}, \quad
h_{2,\text{box}} = \sqrt{\frac{N_{\max}}{s_{\min}\,\mu_{\min}}}, \quad
h_{\text{box}} = \sqrt{\frac{2}{\sqrt{3}}\,\lambda_{\max}\,N_{\max}}$$

The Cartesian product $(2h_{1,\max}+1)^3 \times (2h_{2,\max}+1)^3$ grows as $\sim N_{\max}^3$, making systematic enumeration infeasible beyond $N_{\max} \sim 30$.

In [ ]:
# Show how bounding box scales with Nmax
dil_bounds = (math.sqrt(3)/2, 10.)
moduli_bounds = (2., 5.)
axion_bounds = (-0.5, 0.5)

Nmax_values = [5, 10, 20, 50, 100, 200, 500]
box_sizes = []

for Nm in Nmax_values:
    s = jvc.data_sampler(model, moduli_bounds=moduli_bounds,
        dilaton_bounds=dil_bounds, axion_bounds=axion_bounds, seed=42)
    bf = bounded_fluxes(model, sampler=s, Nmax=Nm)
    bf.compute_eigenvalue_bounds(200, verbose=False)
    dim = bf.dimension_H3
    h1m = int(np.ceil(bf._h1_box))
    h2m = int(np.ceil(bf._h2_box))
    n_box = (2*h1m+1)**dim * (2*h2m+1)**dim
    box_sizes.append(n_box)
    feasible = "✓ enumerate" if n_box < 1e8 else ("⚠ streaming" if n_box < 1e10 else "✗ sample only")
    print(f"Nmax={Nm:>3}: h1_max={h1m:>3}, h2_max={h2m:>2}, "
          f"box={n_box:>18,}  {feasible}")

fig, ax = plt.subplots(figsize=(8, 4),dpi=150)
ax.semilogy(Nmax_values, box_sizes, 'o-', color='steelblue', linewidth=2)
ax.axhline(1e8, color='green', linestyle='--', alpha=0.5, label='enumerate feasible (<10⁸)')
ax.axhline(1e10, color='orange', linestyle='--', alpha=0.5, label='streaming feasible (<10¹⁰)')
ax.set_xlabel(r'$N_{\max}$')
ax.set_ylabel('Bounding box size')
ax.set_title('Search space growth with tadpole bound')
ax.legend()
plt.tight_layout()
plt.show()

## 3. The four sampling methods

| Method | Approach | Completeness | Best regime |
|---|---|---|---|
| **Manual ISD** | Random $(z, \tau, h)$ → ISD complete $f$ → Newton | None | Baseline only |
| **`enumerate_fluxes`** | Systematic: all $h$ in bounding box, stream + filter | Complete | $N_{\max} \lesssim 30$ |
| **`sample_bounded_fluxes`** (Gaussian) | $h \sim \mathcal{N}(0, \sigma^2 M)$ → ISD → Newton | Stochastic | $N_{\max} \gtrsim 50$ |
| **`sample_bounded_fluxes`** (Gaussian + LS) | Same prior + `linearised_shifts` (moves moduli) | Stochastic | Medium $N_{\max}$ |

### Why the Gaussian prior $h \sim \mathcal{N}(0, \sigma^2 M)$?

The continuous ISD tadpole is $N_{\rm flux}^{\rm cont} = s \cdot h^T M^{-1} h$.  For $h$ drawn from $\mathcal{N}(0, \sigma^2 M)$:

$$\mathbb{E}\bigl[s_{\min} \cdot h^T M^{-1} h\bigr] = s_{\min} \cdot \sigma^2 \cdot \mathrm{tr}(I) = s_{\min} \cdot \sigma^2 \cdot d$$

Choosing $\sigma^2 = N_{\max} / (d \cdot s_{\min})$ ensures most samples satisfy $N_{\rm flux} \lesssim N_{\max}$.
The $M$-weighting concentrates samples along eigendirections where $M^{-1}$ has small eigenvalues (low tadpole cost), matching the physical vacuum distribution.

## 4. Benchmark helper

We define a helper function that runs each method with consistent parameters and collects results.

In [ ]:
def run_benchmark(Nmax, moduli_bounds=(2., 5.), dil_bounds=(math.sqrt(3)/2, 10.),
                  axion_bounds=(-0.5, 0.5), n_manual=5000, max_batches_sample=5,
                  n_batch_sample=10_000, skip_enumerate=False):
    """Run all four methods and return a results dict."""
    
    sampler = jvc.data_sampler(model, moduli_bounds=moduli_bounds,
        dilaton_bounds=dil_bounds, axion_bounds=axion_bounds, seed=42)
    results = {}
    
    # ── Method 1: Manual ISD ──
    rng = np.random.default_rng(42)
    t0 = time.perf_counter()
    mod_pts = jnp.array(sampler.get_complex_moduli(n_manual), dtype=complex)
    tau_pts = jnp.array(sampler.get_complex_tau(n_manual), dtype=complex)
    h_all = rng.integers(-5, 6, (n_manual, n_fl))
    
    good = []
    for i in range(n_manual):
        h_i = jnp.array(h_all[i], dtype=float)
        if jnp.all(h_i == 0): continue
        try:
            fl = sampler.ISD_sampling(mod_pts[i], jnp.conj(mod_pts[i]),
                                       tau_pts[i], jnp.conj(tau_pts[i]),
                                       h_i, mode="H")
            if fl is not None and jnp.all(jnp.isfinite(fl)):
                fl_r = jnp.array(fl).real
                fl_r = fl_r.at[:n_fl].set(jnp.round(fl_r[:n_fl]))
                tad = abs(float(jnp.real(model.tadpole(fl_r))))
                if 0 < tad <= Nmax:
                    good.append((mod_pts[i], tau_pts[i], fl_r))
        except: pass
    
    n_conv = 0
    if good:
        bf_tmp = bounded_fluxes(model, sampler=sampler, Nmax=Nmax)
        m_arr = jnp.array([x[0] for x in good], dtype=complex)
        t_arr = jnp.array([x[1] for x in good], dtype=complex)
        f_arr = jnp.array([x[2] for x in good])
        m_o, t_o, r_o = bf_tmp.newton_refine_batch(m_arr, t_arr, f_arr,
            step_size=1.0, tol=1e-10, max_iters=100)
        r_np = np.asarray(r_o)
        in_p = np.asarray(bf_tmp.in_patch_batch(m_o, t_o))
        n_conv = int(np.sum((r_np < 1e-10) & in_p))
    dt = time.perf_counter() - t0
    results['Manual ISD'] = {'vacua': n_conv, 'time': dt,
                              'isd_yield': f"{len(good)}/{n_manual}"}
    
    # ── Method 2: enumerate_fluxes ──
    if not skip_enumerate:
        bf_e = bounded_fluxes(model, sampler=sampler, Nmax=Nmax)
        t0 = time.perf_counter()
        res_e = bf_e.enumerate_fluxes(
            n_sample=200, n_isd_per_h=10, max_h_candidates=5_000_000,
            refine=True, return_moduli=True, newton_tol=1e-10,
            confirm_streaming=False, verbose=False, n_moduli_batches=1)
        dt = time.perf_counter() - t0
        results['enumerate_fluxes'] = {'vacua': len(res_e), 'time': dt}
    
    # ── Method 3: sample_bounded (Gaussian, no LS) ──
    bf_s = bounded_fluxes(model, sampler=sampler, Nmax=Nmax)
    t0 = time.perf_counter()
    res_s = bf_s.sample_bounded_fluxes(
        n_target=500, n_batch=n_batch_sample, n_sample=100, n_mod=10,
        max_batches=max_batches_sample, refine=True, return_moduli=True,
        newton_tol=1e-10, verbose=False)
    dt = time.perf_counter() - t0
    results['sample (Gaussian)'] = {'vacua': len(res_s), 'time': dt}
    
    # ── Method 4: sample_bounded (Gaussian + linearised_shifts) ──
    bf_ls = bounded_fluxes(model, sampler=sampler, Nmax=Nmax)
    t0 = time.perf_counter()
    res_ls = bf_ls.sample_bounded_fluxes(
        n_target=500, n_batch=min(n_batch_sample, 5000), n_sample=100, n_mod=10,
        max_batches=max_batches_sample, refine=True, return_moduli=True,
        newton_tol=1e-10, verbose=False,
        use_linearised_shifts=True, n_isd_iters=5, n_moduli_batches=1)
    dt = time.perf_counter() - t0
    results['sample (Gauss+LS)'] = {'vacua': len(res_ls), 'time': dt}
    
    return results


def print_results(results, Nmax):
    """Pretty-print benchmark results."""
    print(f"\n{'='*65}")
    print(f"  Nmax = {Nmax}")
    print(f"{'='*65}")
    print(f"{'Method':<28} {'Vacua':>7} {'Time':>9} {'Rate':>10}")
    print(f"{'-'*65}")
    for name, r in results.items():
        n = r['vacua']; dt = r['time']
        rate = n / max(dt, 0.1)
        if dt < 120: ts = f"{dt:.1f}s"
        elif dt < 3600: ts = f"{dt/60:.1f}m"
        else: ts = f"{dt/3600:.1f}h"
        extra = f"  (ISD: {r['isd_yield']})" if 'isd_yield' in r else ""
        print(f"{name:<28} {n:>7} {ts:>9} {rate:>9.2f}/s{extra}")
    print(f"{'-'*65}")

## 5. Benchmark: $N_{\max} = 10$ (small tadpole)

At small $N_{\max}$, the bounding box is manageable and `enumerate_fluxes` can systematically check all candidates. The tadpole pre-filter (continuous quadratic form $s_{\min} h^T M^{-1} h \leq N_{\max}$) eliminates >99% of the box, making enumeration fast.

All reported vacua are **Newton-refined** to $\sum_I |D_I W| < 10^{-10}$ and verified to lie inside the sampler's moduli patch.

In [ ]:
res_10 = run_benchmark(Nmax=5, n_manual=5000, max_batches_sample=5, n_batch_sample=10_000)
print_results(res_10, 10)

## 6. Benchmark: $N_{\max} = 200$ (large tadpole)

At large $N_{\max}$, the bounding box is too large for enumeration (~$10^{13}$ candidates). Stochastic sampling with the Gaussian prior becomes the method of choice.

In [ ]:
res_200 = run_benchmark(Nmax=200, skip_enumerate=True,
                        n_manual=5000, max_batches_sample=5, n_batch_sample=10_000)
print_results(res_200, 200)

## 7. Gaussian prior analysis

The Gaussian prior $h \sim \mathcal{N}(0, \sigma^2 M)$ dramatically outperforms uniform sampling because it concentrates samples where the tadpole is small. Let's visualise this.

In [ ]:
# Compare ||h|| distributions: Gaussian prior vs Dataset B
Nmax_plot = 10
s_min_plot = math.sqrt(3) / 2
sampler_plot = jvc.data_sampler(model, moduli_bounds=(2.,5.),
    dilaton_bounds=(s_min_plot, 10.), axion_bounds=(-0.5, 0.5), seed=42)

# Representative ISD matrix
z_rep = jnp.array(sampler_plot.get_complex_moduli(1)[0], dtype=complex)
M_rep = np.asarray(model.ISD_matrix(z_rep, jnp.conj(z_rep)).real)
M_inv_rep = np.linalg.inv(M_rep)

# Gaussian samples
eigvals_M, eigvecs_M = np.linalg.eigh(M_rep)
sigma_sq = Nmax_plot / (n_fl * s_min_plot)
scales = np.sqrt(sigma_sq * np.abs(eigvals_M))
rng_plot = np.random.default_rng(0)
N_samp = 50_000
z_std = rng_plot.standard_normal((N_samp, n_fl))
h_gauss = np.round((z_std * scales[None, :]) @ eigvecs_M.T).astype(int)
h_gauss_norms = np.sqrt(np.sum(h_gauss**2, axis=1))
h_gauss = h_gauss[h_gauss_norms > 0]
h_gauss_norms = h_gauss_norms[h_gauss_norms > 0]

# Tadpole of Gaussian samples
hMh_gauss = np.abs(s_min_plot * np.einsum('hi,ij,hj->h', h_gauss.astype(float), M_inv_rep, h_gauss.astype(float)))

# Uniform-from-box samples (for comparison)
bf_plot = bounded_fluxes(model, sampler=sampler_plot, Nmax=Nmax_plot)
bf_plot.compute_eigenvalue_bounds(200, verbose=False)
h_unif = bf_plot._sample_h_vectors(min(N_samp, 50_000), rng=np.random.default_rng(0))
h_unif_norms = np.sqrt(np.sum(h_unif**2, axis=1))
hMh_unif = np.abs(s_min_plot * np.einsum('hi,ij,hj->h', h_unif.astype(float), M_inv_rep, h_unif.astype(float)))

# Dataset B (if available)
try:
    datsB = jvc.util.load_zipped_pickle("../../../notebooks/dataset_B.p")
    h_B_norms = np.sqrt(np.sum(datsB[:, 12:18]**2, axis=1))
    has_datsB = True
except:
    has_datsB = False

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: ||h|| distribution
bins_h = np.arange(0, 60, 1)
axes[0].hist(h_unif_norms, bins=bins_h, alpha=0.5, density=True, label='Uniform (box)', color='grey')
axes[0].hist(h_gauss_norms, bins=bins_h, alpha=0.6, density=True, label=r'Gaussian $\mathcal{N}(0,\sigma^2 M)$', color='coral')
if has_datsB:
    axes[0].hist(h_B_norms, bins=bins_h, alpha=0.5, density=True, label='Dataset B', color='steelblue')
axes[0].set_xlabel(r'$\|h\|$')
axes[0].set_ylabel('density')
axes[0].set_title(r'$\|h\|$ distribution')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 50)

# Panel 2: Continuous tadpole distribution
bins_t = np.linspace(0, 100, 50)
axes[1].hist(hMh_unif, bins=bins_t, alpha=0.5, density=True, label='Uniform', color='grey')
axes[1].hist(hMh_gauss, bins=bins_t, alpha=0.6, density=True, label='Gaussian', color='coral')
axes[1].axvline(Nmax_plot, color='red', linestyle='--', linewidth=2, label=f'$N_{{\\max}}={Nmax_plot}$')
axes[1].set_xlabel(r'$s_{\min} \cdot h^T M^{-1} h$')
axes[1].set_ylabel('density')
axes[1].set_title('Continuous tadpole')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 100)

# Panel 3: Tadpole pass rate vs Nmax
Nmax_range = np.arange(1, 201)
pass_gauss = [np.sum(hMh_gauss <= nm) / len(hMh_gauss) * 100 for nm in Nmax_range]
pass_unif = [np.sum(hMh_unif <= nm) / len(hMh_unif) * 100 for nm in Nmax_range]
axes[2].plot(Nmax_range, pass_gauss, color='coral', linewidth=2, label='Gaussian')
axes[2].plot(Nmax_range, pass_unif, color='grey', linewidth=2, label='Uniform (box)')
axes[2].set_xlabel(r'$N_{\max}$')
axes[2].set_ylabel('% passing tadpole')
axes[2].set_title('Tadpole acceptance rate')
axes[2].legend()

plt.suptitle(f'Gaussian prior vs uniform box sampling (Nmax={Nmax_plot})', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# Print key numbers
print(f"Tadpole acceptance at Nmax={Nmax_plot}:")
print(f"  Gaussian: {np.sum(hMh_gauss <= Nmax_plot)/len(hMh_gauss)*100:.1f}%")
print(f"  Uniform:  {np.sum(hMh_unif <= Nmax_plot)/len(hMh_unif)*100:.1f}%")
print(f"  → Gaussian is {np.sum(hMh_gauss<=Nmax_plot)/max(np.sum(hMh_unif<=Nmax_plot),1):.0f}× more efficient")

## 8. Summary and recommendations

### Performance by regime

| $N_{\max}$ | Best method | Rate | Notes |
|---|---|---|---|
| $\leq 10$ | `enumerate_fluxes` | ~0.5/s | Systematic, complete coverage |
| $10$–$50$ | `sample_bounded` + LS | ~0.3–6/s | `linearised_shifts` helps find more vacua |
| $\geq 100$ | `sample_bounded` (Gaussian) | **~30/s** | Fastest; LS overhead not worth it |

### Key insights

1. **The Gaussian prior $h \sim \mathcal{N}(0, \sigma^2 M)$ is the single most important improvement.** It concentrates samples where the tadpole is small, giving ~100% acceptance rate vs <1% for uniform box sampling.

2. **`enumerate_fluxes` is complete but slow for large $N_{\max}$.** The bounding box grows as $\sim N_{\max}^3$, and even with the continuous tadpole pre-filter (which eliminates >99% of candidates), the streaming overhead dominates for $N_{\max} > 30$.

3. **`linearised_shifts` helps at small $N_{\max}$ but hurts at large $N_{\max}$.** At small $N_{\max}$, fixed-moduli ISD completion has a low conversion rate (most rounded fluxes don't satisfy the tight tadpole bound), so moving moduli self-consistently via `linearised_shifts` finds more vacua. At large $N_{\max}$, the conversion rate is already high and the ~11ms/h overhead of `linearised_shifts` is wasted.

4. **Manual ISD sampling (random $z$, $\tau$, $h$) is always the worst method.** The ISD completion yield is <2% even at $N_{\max}=200$, because random h-fluxes from a box mostly produce $N_{\rm flux} \gg N_{\max}$ after rounding.

### Recommended workflow

```python
# For small Nmax (complete search):
bf = bounded_fluxes(model, sampler=sampler, Nmax=10)
results = bf.enumerate_fluxes(refine=True, return_moduli=True, verbose=True)

# For large Nmax (fast stochastic search):
bf = bounded_fluxes(model, sampler=sampler, Nmax=200)
results = bf.sample_bounded_fluxes(
    n_target=1000, n_batch=50_000, max_batches=20,
    refine=True, return_moduli=True, verbose=True)
```